# 🏆 Baseline 1: EfficientNetB0 + MiniLM (Late Fusion)
## E-commerce Visual Search — Shopee Dataset (34,250 items) | Google Colab GPU

**Pipeline:**
- 🖼️ **Image Branch:** `EfficientNetB0` → 1280-dim features
- 📝 **Text Branch:** `paraphrase-multilingual-MiniLM-L12-v2` → 384-dim features
- 🔀 **Fusion:** `L2_Normalize(Concat([α × img, (1-α) × txt]))`
- 🔍 **Search:** FAISS `IndexFlatIP` (Cosine Similarity)
- 📊 **Metrics:** mAP@5, Precision@1, Recall@5

**Dataset Split (STRICT — NO DATA LEAKAGE):**
- Gallery : toàn bộ 34,250 ảnh
- Val queries (20%) : ~6,850 → grid search `alpha`
- Test queries (80%) : ~27,400 → đánh giá cuối, chạy **1 lần duy nhất**

## ⚙️ Cell 0: Kiểm tra GPU & Runtime

In [ ]:
# Kiểm tra GPU — nếu thấy 'No GPU' hãy vào Runtime > Change runtime type > T4 GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU khả dụng:')
    print(result.stdout)
else:
    print('❌ Không tìm thấy GPU!')
    print('👉 Vào Runtime > Change runtime type > chọn T4 GPU rồi thử lại!')

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name        : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 📦 Cell 1: Cài đặt thư viện

In [ ]:
# Cài đặt thư viện — chạy 1 lần duy nhất
!pip install -q sentence-transformers faiss-gpu
# torchvision, pandas, numpy, scikit-learn đã có sẵn trên Colab
print('✅ Cài đặt thư viện hoàn tất!')

## 📂 Cell 2: Mount Google Drive & Setup Kaggle

In [ ]:
# ─── CHỌN 1 TRONG 2 CÁCH ĐỌC DATA BÊN DƯỚI ──────────────────────────────────

# ═══ CÁCH A: Đọc từ Google Drive (nếu đã upload dataset lên Drive) ═══════════
USE_DRIVE = False   # ← đổi thành True nếu dùng Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Chỉnh đường dẫn đúng với thư mục dataset của bạn trên Drive
    DATA_DIR = '/content/drive/MyDrive/shopee-product-matching'
    print(f'✅ Đã mount Drive. DATA_DIR = {DATA_DIR}')

# ═══ CÁCH B: Tải từ Kaggle API ════════════════════════════════════════════════
else:
    import os, json

    # Cần file kaggle.json — upload lên Colab hoặc lấy từ Drive
    # Nếu chưa có: vào kaggle.com > Account > Create New API Token
    from google.colab import files
    print('📤 Upload file kaggle.json của bạn:')
    try:
        uploaded = files.upload()   # chọn kaggle.json
        os.makedirs('/root/.config/kaggle', exist_ok=True)
        os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
        os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
        print('✅ kaggle.json đã được cấu hình!')

        # Tải dataset
        print('⏬ Đang tải Shopee dataset từ Kaggle...')
        !pip install -q kaggle
        !kaggle competitions download -c shopee-product-matching -p /content/shopee
        !unzip -q /content/shopee/shopee-product-matching.zip -d /content/shopee
        DATA_DIR = '/content/shopee'
        print(f'✅ Tải xong. DATA_DIR = {DATA_DIR}')
    except Exception as e:
        # Fallback: nếu dataset đã có sẵn ở /content
        print(f'⚠️  Bỏ qua upload: {e}')
        DATA_DIR = '/content/shopee-product-matching'
        print(f'🔧 Dùng DATA_DIR mặc định: {DATA_DIR}')

CSV_PATH = os.path.join(DATA_DIR, 'train.csv')
IMG_DIR  = os.path.join(DATA_DIR, 'train_images')
print(f'\n📄 CSV  : {CSV_PATH}')
print(f'🖼️ Images: {IMG_DIR}')

## 🔧 Cell 3: Import & Cấu hình

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.notebook import tqdm   # dùng tqdm.notebook cho Colab — hiển thị đẹp hơn

import torch
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
import faiss

# ─── Cấu hình ────────────────────────────────────────────────────────────────
BATCH_SIZE  = 128    # T4 GPU ~16GB VRAM → dùng batch lớn hơn cho nhanh
IMG_SIZE    = 224
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
RANDOM_SEED = 42
NUM_WORKERS = 2      # Colab hỗ trợ multiprocessing

print(f'✅ Thiết bị : {DEVICE}')
print(f'✅ Batch size: {BATCH_SIZE}')
print(f'✅ Num workers: {NUM_WORKERS}')

## 📊 Cell 4: Đọc dữ liệu & Chia tập (STRICT SPLIT)

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'📊 Tổng số mẫu : {len(df)}')
print(f'📋 Các cột      : {list(df.columns)}')
print(df.head(3))

# ─── STRICT DATASET SPLIT ────────────────────────────────────────────────────
df_gallery = df.copy()                          # Gallery = toàn bộ dataset

val_idx, test_idx = train_test_split(
    df.index.tolist(),
    test_size   = 0.8,
    random_state= RANDOM_SEED,
    stratify    = df['label_group']             # stratify theo nhãn
)

df_val  = df.loc[val_idx].reset_index(drop=True)
df_test = df.loc[test_idx].reset_index(drop=True)

print(f'\n🗂️  Gallery size  : {len(df_gallery):,} ảnh (toàn bộ dataset)')
print(f'✅ Val queries   : {len(df_val):,} ảnh  → dùng grid search alpha')
print(f'✅ Test queries  : {len(df_test):,} ảnh  → đánh giá cuối (1 lần!)')

## 🖼️ Cell 5: Trích xuất Image Features (EfficientNetB0)

In [ ]:
class ShopeeImageDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.df        = df
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fname = self.df.iloc[idx]['image']
        try:
            img = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        return self.transform(img)


# Transform chuẩn ImageNet
img_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]),
])


@torch.no_grad()
def extract_image_features(df_input, model, transform, img_dir,
                            batch_size=128, num_workers=2):
    dataset = ShopeeImageDataset(df_input, img_dir, transform)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                         num_workers=num_workers, pin_memory=True,
                         persistent_workers=(num_workers > 0))
    all_feats = []
    model.eval()
    for imgs in tqdm(loader, desc='🖼️ Image features'):
        feats = model(imgs.to(DEVICE))          # (B, 1280)
        all_feats.append(feats.cpu().float().numpy())
    return np.vstack(all_feats)


# ─── Tải EfficientNetB0, bỏ classification head ──────────────────────────────
print('⏳ Đang tải EfficientNetB0...')
eff_net = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
eff_net.classifier = torch.nn.Identity()        # bỏ head → output 1280-dim
eff_net = eff_net.to(DEVICE).eval()
print('✅ EfficientNetB0 đã sẵn sàng (output: 1280-dim)')

# Trích xuất gallery image features
print('\n📦 Gallery image features...')
gallery_img_feats = extract_image_features(
    df_gallery, eff_net, img_transform, IMG_DIR, BATCH_SIZE, NUM_WORKERS)
print(f'✅ Gallery img shape: {gallery_img_feats.shape}')

## 📝 Cell 6: Trích xuất Text Features (MiniLM)

In [ ]:
print('⏳ Đang tải MiniLM...')
minilm = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2', device=DEVICE)
print('✅ MiniLM đã sẵn sàng (output: 384-dim)')


def extract_text_features(df_input, model, batch_size=512):
    titles = df_input['title'].fillna('').tolist()
    feats  = model.encode(
        titles,
        batch_size          = batch_size,
        show_progress_bar   = True,
        convert_to_numpy    = True,
        normalize_embeddings= False
    )
    return feats.astype(np.float32)             # (N, 384)


print('\n📦 Gallery text features...')
gallery_txt_feats = extract_text_features(df_gallery, minilm)
print(f'✅ Gallery txt shape: {gallery_txt_feats.shape}')

## 🔀 Cell 7: Late Fusion & FAISS Utils

In [ ]:
def fuse_and_normalize(img_feats, txt_feats, alpha):
    """
    Late Fusion: fused = L2_Normalize(Concat([alpha*img, (1-alpha)*txt]))
    img_feats: (N, 1280) | txt_feats: (N, 384)
    """
    fused = np.concatenate(
        [alpha * img_feats, (1 - alpha) * txt_feats], axis=1)   # (N, 1664)
    norms = np.linalg.norm(fused, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1e-10, norms)
    return (fused / norms).astype(np.float32)


def build_faiss_index(features):
    """FAISS IndexFlatIP — Cosine similarity sau L2-normalize."""
    index = faiss.IndexFlatIP(features.shape[1])
    # Dùng GPU nếu có (nhanh hơn nhiều lần)
    if torch.cuda.is_available():
        res   = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
    index.add(features)
    return index


def get_ground_truth_dict(df_input):
    gt = {}
    for _, grp in df_input.groupby('label_group'):
        ids = set(grp['posting_id'].tolist())
        for pid in ids:
            gt[pid] = ids
    return gt


print('✅ Hàm fuse / FAISS / ground-truth đã sẵn sàng')

## 📐 Cell 8: Hàm đánh giá Retrieval Metrics

In [ ]:
def evaluate_retrieval(query_df, gallery_df, query_img, query_txt,
                       gallery_img, gallery_txt, alpha, K=5):
    """
    Đánh giá retrieval — trả về dict {mAP@K, Precision@1, Recall@K}.
    Self-match (query_posting_id == gallery_posting_id) bị loại bỏ.
    """
    q_fused = fuse_and_normalize(query_img, query_txt, alpha)
    g_fused = fuse_and_normalize(gallery_img, gallery_txt, alpha)

    gt_dict      = get_ground_truth_dict(gallery_df)
    index        = build_faiss_index(g_fused)
    _, indices   = index.search(q_fused, K + 1)     # K+1 để loại self-match
    gallery_pids = gallery_df['posting_id'].tolist()

    ap_list, p1_list, r5_list = [], [], []

    for i, row in enumerate(query_df.itertuples()):
        qid      = row.posting_id
        relevant = gt_dict.get(qid, set()) - {qid}  # loại self-match
        if not relevant:
            continue

        # Top-K (bỏ self-match)
        retrieved = []
        for idx in indices[i]:
            pid = gallery_pids[idx]
            if pid != qid:
                retrieved.append(pid)
            if len(retrieved) == K:
                break

        # AP@K
        hits, ap = 0, 0.0
        for rank, pid in enumerate(retrieved, 1):
            if pid in relevant:
                hits += 1
                ap   += hits / rank
        ap_list.append(ap / min(len(relevant), K))

        # Precision@1
        p1_list.append(1.0 if (retrieved and retrieved[0] in relevant) else 0.0)

        # Recall@K
        r5_list.append(len(set(retrieved) & relevant) / len(relevant))

    return {
        'mAP@5'      : float(np.mean(ap_list)),
        'Precision@1': float(np.mean(p1_list)),
        'Recall@5'   : float(np.mean(r5_list)),
    }


print('✅ Hàm evaluate_retrieval đã sẵn sàng')

## 🔍 Cell 9: Trích xuất Val & Test Features

In [ ]:
# ─── Val features ────────────────────────────────────────────────────────────
print('📦 Val image features...')
val_img_feats = extract_image_features(
    df_val, eff_net, img_transform, IMG_DIR, BATCH_SIZE, NUM_WORKERS)
print('📦 Val text features...')
val_txt_feats = extract_text_features(df_val, minilm)
print(f'✅ Val img: {val_img_feats.shape} | txt: {val_txt_feats.shape}')

# ─── Test features ───────────────────────────────────────────────────────────
print('\n📦 Test image features...')
test_img_feats = extract_image_features(
    df_test, eff_net, img_transform, IMG_DIR, BATCH_SIZE, NUM_WORKERS)
print('📦 Test text features...')
test_txt_feats = extract_text_features(df_test, minilm)
print(f'✅ Test img: {test_img_feats.shape} | txt: {test_txt_feats.shape}')

## 🎯 Cell 10: Grid Search Alpha — Validation Set

In [ ]:
# ⚠️ Grid search CHỈ trên VAL — KHÔNG đụng Test!
alphas = np.arange(0.1, 1.0, 0.1).round(1)
print(f'🔍 Grid search alpha ∈ {alphas.tolist()}')
print('─' * 62)

val_results = []
best_alpha_1, best_map5 = None, -1.0

for alpha in alphas:
    m = evaluate_retrieval(
        query_df    = df_val,
        gallery_df  = df_gallery,
        query_img   = val_img_feats,
        query_txt   = val_txt_feats,
        gallery_img = gallery_img_feats,
        gallery_txt = gallery_txt_feats,
        alpha       = alpha, K=5
    )
    val_results.append({'alpha': alpha, **m})
    marker = ' ← best' if m['mAP@5'] > best_map5 else ''
    print(f'  α={alpha:.1f} | mAP@5={m["mAP@5"]:.4f} | '
          f'P@1={m["Precision@1"]:.4f} | R@5={m["Recall@5"]:.4f}{marker}')
    if m['mAP@5'] > best_map5:
        best_map5    = m['mAP@5']
        best_alpha_1 = alpha

print('─' * 62)
print(f'\n🏆 BEST_ALPHA_1 = {best_alpha_1:.1f}  (Val mAP@5 = {best_map5:.4f})')

## 🧪 Cell 11: Đánh giá TEST SET (Chạy 1 lần duy nhất!)

In [ ]:
# ⚠️  DỪNG LẠI — đọc kỹ trước khi chạy cell này!
# Cell này chỉ được chạy DUY NHẤT 1 LẦN với alpha đã tìm được.
# Chạy lại nhiều lần = data leakage!

print(f'🧪 Đánh giá TEST SET với BEST_ALPHA_1 = {best_alpha_1:.1f}')
print('⚠️  Đây là lần chạy DUY NHẤT trên test set!\n')

test_metrics_1 = evaluate_retrieval(
    query_df    = df_test,
    gallery_df  = df_gallery,
    query_img   = test_img_feats,
    query_txt   = test_txt_feats,
    gallery_img = gallery_img_feats,
    gallery_txt = gallery_txt_feats,
    alpha       = best_alpha_1, K=5
)

print('📊 KẾT QUẢ — Baseline 1 (EfficientNetB0 + MiniLM) trên TEST SET:')
print(f'   mAP@5        = {test_metrics_1["mAP@5"]:.4f}')
print(f'   Precision@1  = {test_metrics_1["Precision@1"]:.4f}')
print(f'   Recall@5     = {test_metrics_1["Recall@5"]:.4f}')

## 📋 Cell 12: Bảng Markdown — Copy vào báo cáo

In [ ]:
from IPython.display import Markdown, display

table = f"""
## 📊 Kết quả so sánh Baseline Models — TEST SET

| Baseline Model | Feature Dim | Best Alpha | Test mAP@5 | Test Precision@1 | Test Recall@5 |
| :--- | :---: | :---: | :---: | :---: | :---: |
| EfficientNetB0 + MiniLM | 1280 + 384 | {best_alpha_1:.1f} | {test_metrics_1['mAP@5']:.4f} | {test_metrics_1['Precision@1']:.4f} | {test_metrics_1['Recall@5']:.4f} |
| MobileCLIP | — | — | — | — | — |

> 💡 Điền kết quả Baseline 2 từ `Baseline2_MobileCLIP.ipynb` vào dòng MobileCLIP.
"""

display(Markdown(table))
print('\n📋 Raw Markdown (copy vào báo cáo):')
print(table)